# 18 - Build ECG-only, CXR-only, combined, and random-control contexts

One fixed, label-coverage-aware external ECG test cohort is used for every condition. CLIP and
CodeBind each retrieve ECG and CXR training records. The same-modality ECG list is the
ECG-to-ECG neighbor RAG baseline motivated by ECG-ReGen; it is not an exact reproduction because
the bridge embeddings and machine diagnoses used here differ. Combined retrieval applies reciprocal-rank
fusion separately within each source modality. A deterministic random-CXR control uses the same
CXR case budget.

Same-modality and cross-modal lists are stored separately before condition construction. This makes
the CXR augmentation effect identifiable and keeps `ecg_plus_cxr_rag` at the same total case budget
as the single-source conditions.


In [ ]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import torch
from tqdm.auto import tqdm

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts"))
RAG_ROOT = ARTIFACT_ROOT / "rag_report_generation_ecg"
config = json.loads((RAG_ROOT / "rag_config.json").read_text())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUERY_BATCH = int(os.environ.get("RAG_RETRIEVAL_QUERY_BATCH", "32"))
CANDIDATE_CHUNK = int(os.environ.get("RAG_RETRIEVAL_CANDIDATE_CHUNK", "50000"))
if QUERY_BATCH < 1 or CANDIDATE_CHUNK < 1:
    raise ValueError("Retrieval batch and chunk sizes must be positive")

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

def file_sha256(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            value = stream.read(chunk)
            if not value:
                break
            digest.update(value)
    return digest.hexdigest()

manifest_paths = {m: ARTIFACT_ROOT / "external" / m / "manifest.jsonl" for m in ["ecg", "cxr"]}
manifests = {m: read_jsonl(path) for m, path in manifest_paths.items()}
concepts = {m: read_jsonl(ARTIFACT_ROOT / "external" / m / "concepts.jsonl") for m in ["ecg", "cxr"]}
for modality in manifests:
    if len(manifests[modality]) != len(concepts[modality]):
        raise ValueError(f"{modality} manifest/concept length mismatch; rerun notebook 14")
    for index, (row, concept) in enumerate(zip(manifests[modality], concepts[modality])):
        if str(row["sample_id"]) != str(concept.get("sample_id")):
            raise ValueError(f"{modality} concept order mismatch at {index}")
manifest_hashes = {m: file_sha256(path) for m, path in manifest_paths.items()}
print({m: {s: sum(row.get("split") == s for row in values) for s in ["train", "val", "test"]}
       for m, values in manifests.items()})


In [ ]:
selections = config["retrieval_selection"]
embeddings, status = {}, {}
for method, experiment in selections.items():
    embeddings[method] = {}
    for modality in ["ecg", "cxr"]:
        path = (ARTIFACT_ROOT / "external" / modality / "bridge" /
                experiment["variant"] / f"seed_{experiment['seed']}" / "common.npy")
        array = np.load(path, mmap_mode="r")
        if len(array) != len(manifests[modality]):
            raise ValueError(f"{method}/{modality} embeddings do not match the manifest")
        embeddings[method][modality] = array
for modality in ["ecg", "cxr"]:
    status[modality] = np.load(
        ARTIFACT_ROOT / "external" / modality / "encoding_status.npy", mmap_mode="r")
    if len(status[modality]) != len(manifests[modality]):
        raise ValueError(f"{modality} status does not match the manifest")

def finite_row_mask(array, chunk_size=50000):
    mask = np.zeros(len(array), dtype=bool)
    for start in tqdm(range(0, len(array), chunk_size), desc="validate embeddings", leave=False):
        stop = min(start + chunk_size, len(array))
        mask[start:stop] = np.isfinite(np.asarray(array[start:stop])).all(axis=1)
    return mask

split_masks = {
    modality: {split: np.asarray([row.get("split") == split for row in rows], dtype=bool)
               for split in ["train", "test"]}
    for modality, rows in manifests.items()
}
valid_masks = {
    method: {modality: (np.asarray(status[modality]) == 1) & finite_row_mask(array)
             for modality, array in values.items()}
    for method, values in embeddings.items()
}
joint_valid = {
    modality: np.logical_and.reduce([valid_masks[method][modality] for method in embeddings])
    for modality in ["ecg", "cxr"]}


In [ ]:
def stable_digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()

def ecg_positive_labels(index):
    return {str(value).split(":", 1)[1] for value in concepts["ecg"][index].get("concepts", [])
            if str(value).startswith("present:")}

rng = np.random.default_rng(int(config["seed"]))
eligible = np.flatnonzero(split_masks["ecg"]["test"] & joint_valid["ecg"]).astype("int64")
if len(eligible) < 2:
    raise ValueError(f"Need at least two valid external ECG test queries; found {len(eligible)}")
take = min(int(config["max_test_ecg"]), len(eligible))
shuffled = eligible[rng.permutation(len(eligible))]
label_to_indices = {}
for index in shuffled:
    for label in ecg_positive_labels(int(index)):
        label_to_indices.setdefault(label, []).append(int(index))
selected_indices, selected_set = [], set()
for label in sorted(label_to_indices, key=lambda value: (len(label_to_indices[value]), value)):
    candidate = next((index for index in label_to_indices[label] if index not in selected_set), None)
    if candidate is not None and len(selected_indices) < take:
        selected_indices.append(candidate); selected_set.add(candidate)
for index in shuffled:
    if len(selected_indices) >= take:
        break
    if int(index) not in selected_set:
        selected_indices.append(int(index)); selected_set.add(int(index))
cohort = np.asarray(sorted(selected_indices), dtype="int64")
selected_labels = sorted(set().union(*(ecg_positive_labels(int(i)) for i in cohort)))
cohort_signature = stable_digest({
    "manifest_hashes": manifest_hashes,
    "seed": config["seed"], "selected_ids": [manifests["ecg"][int(i)]["sample_id"] for i in cohort],
    "condition_specs": config["condition_specs"],
    "context_cases": config["context_cases"],
    "retrieval_selection": {key: {"variant": value["variant"], "seed": value["seed"]}
                            for key, value in selections.items()},
    "cxr_context_content": config["cxr_context_content"],
})
coverage = {
    "available_positive_labels": sorted(label_to_indices),
    "selected_positive_labels": selected_labels,
    "missing_positive_labels": sorted(set(label_to_indices) - set(selected_labels)),
}
print({"queries": len(cohort), "signature": cohort_signature[:12], "label_coverage": coverage})


In [ ]:
def numeric_subject(row):
    try:
        return int(row.get("subject_id")) if row.get("subject_id") is not None else -1
    except (TypeError, ValueError):
        return -1

@torch.inference_mode()
def chunked_topk(query_values, candidate_values, query_rows, candidate_rows, k):
    k = min(int(k), len(candidate_values))
    if k < 1:
        return [[] for _ in query_rows]
    candidate_subjects = np.asarray([numeric_subject(row) for row in candidate_rows], dtype="int64")
    outputs = []
    for q_start in tqdm(range(0, len(query_values), QUERY_BATCH), desc="query batches", leave=False):
        q_stop = min(q_start + QUERY_BATCH, len(query_values))
        q = torch.from_numpy(np.asarray(query_values[q_start:q_stop], dtype="float32")).to(DEVICE)
        q = torch.nn.functional.normalize(q, dim=-1)
        q_subjects = np.asarray([numeric_subject(row) for row in query_rows[q_start:q_stop]], dtype="int64")
        best_scores = torch.full((len(q), k), -torch.inf, device=DEVICE)
        best_indices = torch.full((len(q), k), -1, dtype=torch.long, device=DEVICE)
        for c_start in range(0, len(candidate_values), CANDIDATE_CHUNK):
            c_stop = min(c_start + CANDIDATE_CHUNK, len(candidate_values))
            c = torch.from_numpy(np.asarray(candidate_values[c_start:c_stop], dtype="float32")).to(DEVICE)
            c = torch.nn.functional.normalize(c, dim=-1)
            scores = q @ c.T
            c_subjects = candidate_subjects[c_start:c_stop]
            same_subject = ((q_subjects[:, None] >= 0) &
                            (q_subjects[:, None] == c_subjects[None, :]))
            if same_subject.any():
                scores.masked_fill_(torch.from_numpy(same_subject).to(DEVICE), -torch.inf)
            local_k = min(k, scores.shape[1])
            local_scores, local_positions = scores.topk(local_k, dim=1)
            merged_scores = torch.cat([best_scores, local_scores], dim=1)
            merged_indices = torch.cat([best_indices, local_positions + c_start], dim=1)
            best_scores, order = merged_scores.topk(k, dim=1)
            best_indices = torch.gather(merged_indices, 1, order)
        for row_scores, row_indices in zip(best_scores.cpu().numpy(), best_indices.cpu().numpy()):
            if (row_indices < 0).any() or not np.isfinite(row_scores).all():
                raise RuntimeError("Insufficient candidates after patient leakage masking")
            outputs.append([(int(i), float(score)) for i, score in zip(row_indices, row_scores)])
    return outputs

def retrieval_text(modality, row):
    key = "machine_diagnosis" if modality == "ecg" else "report_text"
    return " ".join(str(row.get(key, "")).split())

def normalized_labels(row):
    return sorted({str(value).strip() for value in row.get("labels", []) if str(value).strip()})

def item(method, scope, rank, score, modality, row):
    return {
        "retrieval_method": method, "scope": scope, "rank": int(rank), "score": float(score),
        "modality": modality, "sample_id": str(row["sample_id"]),
        "subject_id": row.get("subject_id"), "text": retrieval_text(modality, row),
        "labels": normalized_labels(row) if modality == "cxr" else [],
    }


In [ ]:
query_rows = [manifests["ecg"][int(i)] for i in cohort]
contexts_by_method = {method: {} for method in ["clip", "codebind"]}
diagnostics = {"cohort_signature": cohort_signature, "device": str(DEVICE), "retrievals": {}}
for method in ["clip", "codebind"]:
    method_records = {str(row["sample_id"]): {"same_modality": [], "cross_modal": []}
                      for row in query_rows}
    for scope, candidate_modality in [("same_modality", "ecg"), ("cross_modal", "cxr")]:
        # Use the same finite training candidate pool for every retriever so a
        # CLIP/CodeBind difference cannot be caused by different eligibility.
        candidate_indices = np.flatnonzero(
            split_masks[candidate_modality]["train"] & joint_valid[candidate_modality]
        ).astype("int64")
        candidate_rows = [manifests[candidate_modality][int(i)] for i in candidate_indices]
        found = chunked_topk(
            embeddings[method]["ecg"][cohort],
            embeddings[method][candidate_modality][candidate_indices],
            query_rows, candidate_rows, int(config["context_cases"]))
        for query_row, values in zip(query_rows, found):
            key = str(query_row["sample_id"])
            for rank, (position, score) in enumerate(values, start=1):
                source = candidate_rows[position]
                if source.get("split") != "train":
                    raise AssertionError("Non-training retrieval candidate")
                if str(source["sample_id"]) == key:
                    raise AssertionError("Query sample leaked into retrieval context")
                q_subject, c_subject = numeric_subject(query_row), numeric_subject(source)
                if q_subject >= 0 and q_subject == c_subject:
                    raise AssertionError("Query subject leaked into retrieval context")
                method_records[key][scope].append(
                    item(method, scope, rank, score, candidate_modality, source))
        diagnostics["retrievals"][f"{method}:{scope}"] = {
            "queries": len(query_rows), "candidate_modality": candidate_modality,
            "training_candidates": len(candidate_indices), "top_k": int(config["context_cases"]),
        }
    contexts_by_method[method] = method_records
    print(method, {scope: len(contexts_by_method[method][str(query_rows[0]['sample_id'])][scope])
                   for scope in ["same_modality", "cross_modal"]})


In [ ]:
def reciprocal_rank_fusion(left, right, method="combined", constant=60):
    fused = {}
    for value in left + right:
        key = (value["modality"], value["sample_id"])
        entry = fused.setdefault(key, {**value, "score": 0.0, "retrieval_method": method})
        entry["score"] += 1.0 / (constant + int(value["rank"]))
    ordered = sorted(fused.values(), key=lambda row: (-row["score"], row["sample_id"]))
    for rank, value in enumerate(ordered, start=1):
        value["rank"] = rank
    return ordered

combined_contexts = {}
for query_row in query_rows:
    key = str(query_row["sample_id"])
    combined_contexts[key] = {}
    for scope in ["same_modality", "cross_modal"]:
        combined_contexts[key][scope] = reciprocal_rank_fusion(
            contexts_by_method["clip"][key][scope],
            contexts_by_method["codebind"][key][scope])[:int(config["context_cases"])]

random_candidate_indices = np.flatnonzero(
    split_masks["cxr"]["train"] & joint_valid["cxr"]).astype("int64")
random_candidate_rows = [manifests["cxr"][int(i)] for i in random_candidate_indices]
random_contexts = {}
for query_row in query_rows:
    key = str(query_row["sample_id"])
    q_subject = numeric_subject(query_row)
    eligible_positions = np.asarray([
        position for position, row in enumerate(random_candidate_rows)
        if q_subject < 0 or numeric_subject(row) != q_subject], dtype="int64")
    if len(eligible_positions) < int(config["context_cases"]):
        raise RuntimeError("Insufficient random CXR candidates after patient exclusion")
    local_seed = int(hashlib.sha1(
        f"{config['seed']}:{key}:random_cxr".encode()).hexdigest()[:16], 16)
    local_rng = np.random.default_rng(local_seed)
    positions = local_rng.choice(
        eligible_positions, size=int(config["context_cases"]), replace=False)
    random_contexts[key] = [
        item("random", "cross_modal", rank, 0.0, "cxr", random_candidate_rows[int(position)])
        for rank, position in enumerate(positions, start=1)]


In [ ]:
spec_lookup = {row["condition"]: row for row in config["condition_specs"]}

def selected_context(spec, sample_id):
    method, scope = spec["retrieval_method"], spec["context_scope"]
    if method == "none":
        return []
    if method == "random":
        values = random_contexts[sample_id][:int(spec["cxr_cases"])]
    else:
        source = combined_contexts[sample_id] if method == "combined" else contexts_by_method[method][sample_id]
        values = []
        if scope in {"ecg_only", "ecg_plus_cxr"}:
            values.extend(source["same_modality"][:int(spec["ecg_cases"])])
        if scope in {"cxr_only", "ecg_plus_cxr"}:
            values.extend(source["cross_modal"][:int(spec["cxr_cases"])])
    expected = int(spec["ecg_cases"]) + int(spec["cxr_cases"])
    if len(values) != expected:
        raise AssertionError(f"Context budget mismatch for {spec['condition']}: {len(values)} != {expected}")
    return values

context_rows = []
for query_index, query_row in zip(cohort, query_rows):
    sample_id = str(query_row["sample_id"])
    common = {
        "sample_id": sample_id, "modality": "ecg", "query_index": int(query_index),
        "query_path": str(query_row["waveform_path"]), "split": str(query_row["split"]),
        "cohort_signature": cohort_signature,
    }
    for condition in config["conditions"]:
        spec = spec_lookup[condition]
        context_rows.append({
            **common, **spec, "retrieved": selected_context(spec, sample_id),
            "cxr_context_content": config["cxr_context_content"],
        })

output = RAG_ROOT / "contexts" / "rag_contexts.jsonl"
temporary = output.with_suffix(".jsonl.tmp")
with temporary.open("w") as stream:
    for row in context_rows:
        stream.write(json.dumps(row) + "\n")
temporary.replace(output)
diagnostics.update({
    "rows": len(context_rows), "conditions": config["conditions"],
    "queries": len(cohort), "condition_case_budgets": {
        row["condition"]: row["ecg_cases"] + row["cxr_cases"] for row in config["condition_specs"]},
    "cohort_label_coverage": coverage, "leakage_violations": 0,
    "random_cxr_training_candidates": len(random_candidate_rows),
})
(RAG_ROOT / "contexts" / "retrieval_diagnostics.json").write_text(
    json.dumps(diagnostics, indent=2))
print("Saved:", output, diagnostics)
